# Streaming and error states

Building conversational or generative UIs requires managing high-frequency data updates without locking up the browser thread. Traditional web apps handle payloads as a single atomic JSON block. GenAI apps handle continuous streams of token deltas, requiring specialized frontend strategies.

## 1. Core Concepts & Engineering Mechanics
**The Re-Render Storm (DOM Thrashing):**
LLM endpoints can stream tokens at rates exceeding 50 to 100 tokens per second. If a React component triggers a full state change and layout re-calculation for every individual chunk, the JavaScript main thread freezes, causing severe input lag and jitter.

Production Fix: Implement requestAnimationFrame throttling or time-based batching (e.g., flushing state updates every 50ms) to group incoming chunks smoothly.

**Optimistic Local State & Rollbacks:**
When a user submits a prompt, waiting for the HTTP roundtrip before showing the message in the chat feed introduces sluggish UX. Production apps write the message to local state optimistically right away, assigning a temporary client-side ID while tracking a network sync flag.

**Mid-Stream Network Drops & SSE Resiliency:**
Because token streaming uses Server-Sent Events (SSE) over persistent HTTP connections, transient network blips can cut off a stream halfway through a paragraph. The client must catch the socket drop, preserve the partial text accumulated so far, and render a recovery action state rather than crashing the UI component tree.

## 2. Production Implementation Pattern (Throttled Stream State Management)
Here is a clean frontend architectural pattern (TypeScript / React hook model) showing how to consume an SSE text stream safely with batched state throttling to prevent browser lockup.

In [ ]:
import { useState, useRef, useCallback } from 'react';

export function useThrottledStream() {
  const [displayText, setDisplayText] = useState<string>('');
  const [isStreaming, setIsStreaming] = useState<boolean>(false);
  
  // Ref to hold buffer without triggering immediate React re-renders on every token
  const bufferRef = useRef<string>('');
  const frameIdRef = useRef<number | null>(null);

  // Throttled UI flusher using requestAnimationFrame
  const flushBuffer = useCallback(() => {
    setDisplayText(bufferRef.current);
    frameIdRef.current = null;
  }, []);

  const appendToken = useCallback((token: string) => {
    bufferRef.current += token;
    
    // Schedule render flush only if not already pending
    if (!frameIdRef.current) {
      frameIdRef.current = requestAnimationFrame(flushBuffer);
    }
  }, [flushBuffer]);

  const startStreamSimulation = async (endpointUrl: string, payload: object) => {
    setIsStreaming(true);
    bufferRef.current = '';
    setDisplayText('');

    try {
      const response = await fetch(endpointUrl, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(payload),
      });

      if (!response.ok || !response.body) throw new Error('Network response failed');

      const reader = response.body.getReader();
      const decoder = new TextDecoder();

      while (true) {
        const { value, done } = await reader.read();
        if (done) break;
        
        const chunk = decoder.decode(value, { stream: true });
        appendToken(chunk);
      }
    } catch (error) {
      console.error("Stream interrupted:", error);
      bufferRef.current += "\n[Connection interrupted. Partial response preserved.]";
      flushBuffer();
    } finally {
      setIsStreaming(false);
      if (frameIdRef.current) cancelAnimationFrame(frameIdRef.current);
      flushBuffer(); // Final hard sync flush
    }
  };

  return { displayText, isStreaming, startStreamSimulation };
}

## 3. Deep-Dive: Architecture & Frontend Trade-offs
**Why WebSockets are Often Overrated for Standard LLMs:** Why standard GenAI apps use SSE instead of WebSockets. Emphasize that unidirectional SSE runs natively over HTTP/2, easily traverses enterprise firewalls and proxies, handles auto-reconnection out of the box, and requires zero custom messaging protocols compared to maintaining stateful WebSocket servers.

**Auto-Scrolling vs. User Intent Control:** In real-time chat apps, auto-scrolling the viewport down as new tokens stream in is standard—unless the user manually scrolls upward to read past messages. Production chat UIs must implement a scroll-position observer that halts auto-scrolling when the user overrides it, preventing frustrating layout jumps.